# EchoProof P1-13: freeze and reproduce the Colab baseline

Use this same notebook in two separate Colab VMs. Stage A exports the successful environment. Stage B recreates and verifies it after **Disconnect and delete runtime**, followed by a fresh GPU connection. An ordinary kernel restart is insufficient.

Sync `pyproject.toml`, `src/echoproof/`, `scripts/p1_13_colab_verify.py`, all `tests/test_p1_*.py`, `configs/`, and this notebook to Drive **before Stage A**. Keep the existing `runs/phase1/` directories. Read `docs/environment_reproduction.md`. Do not edit source/config/test/script files between stages: their hashes must match.

This covers LR/BiGRU baseline reproduction. It does not train on speech or close the deferred pretrained gate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import importlib.util
import json

PROJECT_ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')
STAGE = 'export'  # Stage A: export. After a fresh VM, change ONLY this notebook setting to reproduce.
EXPORT_DIR = None  # Stage B: paste the exact Stage A export_dir string here.

assert STAGE in ('export', 'reproduce')
assert (PROJECT_ROOT / 'scripts/p1_13_colab_verify.py').is_file()
spec = importlib.util.spec_from_file_location('p1_13_colab_verify', PROJECT_ROOT / 'scripts/p1_13_colab_verify.py')
p113 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(p113)

## Stage A: export from the successful GPU environment

Select a GPU runtime before export. If the P1-12 runtime is still active, run this stage there. If it has already been reclaimed, restore and rerun the P1-12 notebook first; export then describes that restored environment rather than proving the earlier package state.

The export records exact baseline dependency pins, a broader package inventory, Python/build/decoder/GPU details, PyTorch distribution metadata, source hashes, and historical artifact hashes. Any unresolved dependency, unsupported local/direct baseline package origin or broken historical manifest stops export. An official PyTorch wheel origin is retained when present. No Windows virtual environment is copied.

Stage A automatically installs `jedi` if missing, records the repair, confirms the PyTorch version is unchanged and requires `pip check` to pass before export. If a previous export attempt failed, keep that report and rerun this stage after syncing the updated runner; a new report/export path is created.

In [ ]:
if STAGE == 'export':
    export_report, export_evidence = p113.export(PROJECT_ROOT)
    print(json.dumps(export_report, indent=2))
    if export_report['status'] != 'exported_for_reproduction':
        raise RuntimeError('Export needs attention. Read the printed report; do not delete the runtime yet.')
    print('SAVE THIS EXPORT_DIR:', export_report['export_dir'])
    print('Verify the lock directory and export report are visible in Drive before disconnecting.')

## Stage B: reconnect to a fresh GPU VM

1. Save the Stage A `export_dir` value and wait for its files to appear in Drive.
2. Use **Disconnect and delete runtime**, then reconnect with a GPU. Reopen this notebook if necessary.
3. Set `STAGE = 'reproduce'` and paste `EXPORT_DIR`; rerun the mount/settings cell and this stage. Do not run P1-03 or import NumPy/PyTorch into the notebook first.

The runner confirms a different VM boot identity, matching Python/platform and preserved files before installation. It installs differing exact pins, installs EchoProof without dependency resolution/build isolation, checks the environment and `pip check`, runs baseline tests/probes plus a CUDA BiGRU forward/backward/AdamW check in new Python processes, and records final verification. A matching preinstalled torch build is retained. A missing/different torch build is installed from the recorded official wheel channel; unavailable exact wheels cause a visible failure. Package downloads may be large; they use the VM's temporary package/cache storage.

If Python or FFmpeg changed, review the mismatch rather than changing an immutable export. If the GPU model changed, the report requires budget reprofiling. A same-VM rerun is rejected. This notebook does not infer available work hours or current Drive quota.

In [ ]:
if STAGE == 'reproduce':
    if not EXPORT_DIR:
        raise ValueError('Paste the exact Stage A export_dir into EXPORT_DIR.')
    reproduction_report, reproduction_evidence = p113.reproduce(PROJECT_ROOT, EXPORT_DIR, install=True)
    print(json.dumps({k:v for k,v in reproduction_report.items() if k != 'checks'}, indent=2))
    if reproduction_report['status'] != 'ready_for_review':
        print(json.dumps(reproduction_report['checks'], indent=2))
        raise RuntimeError('Reproduction needs attention. Preserve the report and inspect the failed checks.')

## Handoff

Share the printed reproduction evidence folder's `verification.json` and `verification.sha256`, together with the exported lock directory. New baseline probe runs are saved under `runs/phase1`; their environment records carry the dependency-lock hash. Earlier run records remain unchanged.

`ready_for_review` requires an actual fresh VM, matching exact packages/runtime, preserved source/artifacts, passing dependency checks, and all selected tests/probes passing without skips. Independent M/U review, a hardware-triggered budget reprofile, and P1-14 handoff are recorded separately. Offline installation and interrupted training continuation are not established by this baseline check.